# NeuroPET image-registration workshop — facilitator notebook

This is the click-and-run answer key, timing guide, and technical check for `NeuroPET_exercise.ipynb`. Run all cells before the session.

> All images are educational simulations built from a BrainWeb digital phantom, not patient scans.

## RUN THIS — pre-flight check

In [ ]:
import platform
import matplotlib
import numpy as np
from workshop_helpers import (
    compare_cases, how_well_matched, load_workshop_data,
    measure_gm_uptake, optimise, shift_image, show_alignment,
)

data = load_workshop_data()
mri = data["mri"]
gm_mask = data["gm_mask"]
gm_contour = data["gm_probability"]
low_pet = data["low_binding_pet"]
high_pet = data["high_binding_pet"]
challenge_pet = data["challenge_pet"]
print(f"Python {platform.python_version()} | NumPy {np.__version__} | Matplotlib {matplotlib.__version__}")
print(f"Dataset: {mri.shape}, BrainWeb Python {data['metadata']['brainweb_python_version']}")
print("✓ Pre-flight imports and data load succeeded.")

## Suggested 50-minute run sheet

| Time | Student section | Speaking prompt |
|---:|---|---|
| 0–5 | Welcome | Anatomy image versus tracer-distribution image |
| 5–8 | Setup | Everyone runs the same completed code |
| 8–13 | Meet images | What does each panel communicate? |
| 13–23 | Manual registration | One axis at a time; visual reasoning first |
| 23–30 | Match score | A computer needs a numerical definition of better |
| 30–38 | Optimiser | 441 candidates; what happens with rotation? |
| 38–46 | Measurement | Wrong alignment means wrong pixels |
| 46–50 | Compare + recap | Imaging contributes evidence; it is not a diagnosis |

Allow 5–10 extra minutes for discussion in a one-hour session.

## Answer key — manual and automatic registration

The challenge image was displaced **7 pixels right and 5 pixels up**. Therefore the correction is **x = −7, y = +5**. Positive x moves content right; positive y moves it down.

In [ ]:
initial_score = how_well_matched(mri, challenge_pet)
manual_answer = shift_image(challenge_pet, x_offset=-7, y_offset=5)
manual_score = how_well_matched(mri, manual_answer)
automatic_pet, best_x, best_y, automatic_score = optimise(mri, challenge_pet, search_range=10)
print(f"Initial NMI:    {initial_score:.3f}")
print(f"Manual answer:  {manual_score:.3f} at x=-7, y=+5")
print(f"Automatic:      {automatic_score:.3f} at x={best_x:+d}, y={best_y:+d}")
assert (best_x, best_y) == (-7, 5)
assert automatic_score > initial_score
show_alignment(mri, automatic_pet, gm_contour, title="Expected automatic result");

### Teaching note

The score is normalized mutual information (NMI), ranging from 0 to 1. It measures statistical dependence between binned MRI and PET intensities, so corresponding anatomy may have different brightness in the two modalities. Call it a **match score**, never percentage accuracy. Raw pixel subtraction and ordinary correlation are less suitable here because MRI and PET intensities mean different things.

## Answer key — uptake measurement

In [ ]:
bad_uptake = measure_gm_uptake(challenge_pet, gm_mask)
registered_uptake = measure_gm_uptake(automatic_pet, gm_mask)
figure, low_uptake, high_uptake = compare_cases(low_pet, high_pet, gm_mask)
print(f"Misregistered low-binding measurement: {bad_uptake:.3f}")
print(f"Registered low-binding measurement:    {registered_uptake:.3f}")
print(f"Low-binding aligned case:              {low_uptake:.3f}")
print(f"High-binding aligned case:             {high_uptake:.3f}")
assert registered_uptake > bad_uptake
assert high_uptake > low_uptake

## Interpretation language to model

- The high-binding **synthetic** case contains more simulated cortical grey-matter tracer retention.
- In an amyloid-PET context, increased cortical retention can contribute evidence of amyloid pathology.
- One PET measurement does **not** diagnose Alzheimer's disease. Real assessment also requires appropriate patient selection, expert interpretation, symptoms, history, examination, cognitive assessment, and other clinical information.
- The mask is grey-matter tissue, not a validated precuneus, posterior-cingulate, or temporal-lobe region.

## Troubleshooting

- **A student gets lost:** Kernel → Restart Kernel and Run All, then return to the two-number cell. Its initial zero values are valid.
- **Plots appear twice:** Keep the semicolon at the end of plotting cells.
- **Data file missing locally:** run `python prepare_brainweb_data.py` once. Binder runs this during image build.
- **Internet unavailable in class:** launch Binder instances beforehand or prepare the data locally. Student notebook execution itself makes no network requests.
- **Discussion runs short:** ask how adding rotation changes the search space, or why a high score does not prove registration is clinically correct.

See `DATA_SOURCES.md` for data provenance and the unresolved BrainWeb redistribution boundary.